# Nile Red: pixelated relative pixel 3 QE

Nile Red is a solvatochromic dye: its emission spectrum shifts with the polarity of its local environment, so
where it binds reports on local chemistry as well as position. A spectral shift changes how a molecule's
photoelectrons divide between the Bayer pixel types, and for Nile Red almost all of the change is between the
red (pixel 3) and green (pixel 2) pixels. This notebook maps that directly, with no spectral model: it bins the
localisations onto a spatial grid (50 nm cells) and gives each cell its **relative pixel 3 QE**,

$$f_R = \frac{\sum_i N_i A_{R,i}}{\sum_i N_i (A_{R,i} + A_{G,i} + A_{B,i})},$$

the share of the cell's summed photoelectrons ($N_i$ per localisation, split by its fitted relative pixel QEs
$A_{p,i}$) that fell on pixel 3. The result is a spatial *map* of local environment, not a per-molecule identity
like `unmix_channels` in the previous notebook. A change in $f_R$ can only come from a change in the emission
spectrum; no wavelength is assigned.

Data: `test_tiffs/nile_red_example/`. **No raw TIFF** (see its `README.md`): a 60,000-frame,
0.05 localisations/µm² PAINT acquisition was simulated, fitted and clustered through the real pipeline, and
only the fitted `nile_red_example_localisations.h5` ships. Two spatially separated Nile Red environments with
different emission spectra were simulated (the ground truth labels them by their simulated spectra, 620 nm and
640 nm centre of mass), with dense, overlapping coverage, since each grid cell needs several localisations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

import pyS3M.IOFunctions as IOFunctions
import pyS3M.NileRedFunctions as NileRedFunctions
import pyS3M.PlottingBase as PlottingBase

REPO_ROOT = Path.cwd().resolve().parents[1]
DATA_DIR = REPO_ROOT / "test_tiffs" / "nile_red_example"
LOCS_PATH = DATA_DIR / "nile_red_example_localisations.h5"
TRUTH_PATH = DATA_DIR / "ground_truth" / "nile_red_example_ground_truth.h5"
CAMERA_PIXEL_NM = 69.0   # Ximea pixel in the sample plane: the fixture was simulated on it

plotter = PlottingBase.PublicationPlotter()
locs = IOFunctions.IO_Functions().read_h5_database(str(LOCS_PATH))
print(f"{len(locs):,} fitted localisations")

## Pixelate

`NileRed_Functions.pixelate_relative_pixel3_qe` (the same function the GUI's Nile Red panel runs) adds each
localisation's photoelectrons, split by $A_R, A_G, A_B$, to the grid cell it falls in. Cells with fewer than
`MIN_LOCALISATIONS` localisations are left empty (NaN); localisations without finite relative pixel QEs or
photoelectrons are left out.

In [ ]:
GRID_NM = 50.0
MIN_LOCALISATIONS = 3

nile_red = NileRedFunctions.NileRed_Functions()
locs_px, grid_info = nile_red.pixelate_relative_pixel3_qe(
    locs, pixel_size_nm=GRID_NM, camera_pixel_size=CAMERA_PIXEL_NM, min_localisations=MIN_LOCALISATIONS)
pixel3_map = grid_info["pixel3_grid"]
iy, ix = np.nonzero(np.isfinite(pixel3_map))
cells = pd.DataFrame({"ix": ix, "iy": iy, "f_R": pixel3_map[iy, ix]})
print(f"{grid_info['n_pixels']:,} grid cells with at least {MIN_LOCALISATIONS} localisations "
      f"({grid_info['n_pixels_skipped']:,} occupied cells below that)")

## Check against the ground truth

Each cell is assigned the environment of the nearest simulated molecule to its centre. The two environments
give two separate distributions of $f_R$: the redder environment puts a larger share of its photoelectrons on
pixel 3.

In [ ]:
truth = pd.read_hdf(TRUTH_PATH)
centres_nm = np.asarray(grid_info["origin_nm"]) + (cells[["ix", "iy"]].to_numpy() + 0.5) * GRID_NM
_, nearest = cKDTree(truth[["xc_nm", "yc_nm"]].to_numpy()).query(centres_nm)
cells["environment"] = truth["dye"].to_numpy()[nearest]

fig, ax = plotter.create_figure()
bins = np.linspace(cells["f_R"].quantile(0.005), cells["f_R"].quantile(0.995), 60)
for (env, g), colour in zip(cells.groupby("environment"), ["tab:orange", "firebrick"]):
    ax.hist(g["f_R"], bins=bins, color=colour, alpha=0.6, label=f"{env}: {g['f_R'].median():.3f} (n = {len(g):,})")
    ax.axvline(g["f_R"].median(), color=colour, ls="--", lw=1)
ax.set_xlabel("relative pixel 3 QE")
ax.set_ylabel("grid cells")
ax.grid(True, ls="--", color="grey", alpha=0.5)
ax.set_axisbelow(True)
ax.legend(fontsize=7, loc="upper center")
plt.show()

## Spatial map

The pixelated map itself, drawn as an image: its cost is set by the grid size, not by the number of
localisations. The two environments appear as two regions of different relative pixel 3 QE.

In [ ]:
lo, hi = np.nanpercentile(pixel3_map, [1, 99])
fig, ax = plotter.create_figure()
plotter.image_plot(ax, pixel3_map, cmap="turbo", vmin=lo, vmax=hi, origin="lower",
                   show_axes=True, xlabel=f"grid x ({GRID_NM:g} nm)", ylabel=f"grid y ({GRID_NM:g} nm)",
                   colorbar=True, colorbar_label="relative pixel 3 QE")
plt.show()

## Next: `07_frc.ipynb`

This fixture's dense, repeat-sampled localisations are also the ones `07_frc.ipynb` uses to demonstrate a
resolved FRC curve.